# Datasets Pre-processing

This Jupyter notebook implements a data preprocessing pipeline for four Portuguese sentiment analysis datasets (IMDB-PT, OpCovidBR, ReLI, and TweetSentBR). The notebook is designed to:

1. Reading the files in their original formats as DataFrames;

2. Removing unnecessary columns, keeping only the texts and their respective labels in the DataFrame

3. Removing instances that are not of the `Positive`, `Negative` or `Neutral` class;

4. Standardizing the names of the remaining columns;

5. Standardizing the class labels: `1` for the `Positive` sentiment, `-1` for the `Negative` sentiment and `0` for `Neutral` sentiment;

6. Splitting the dataset into training and test subsets;

7. Saving the training and test subsets in CSV format;

The notebook is designed to run locally or in any Jupyter-compatible environment and requires environment variables defined in a `.env` file (see `.env.example` at the project root).


<br>

**Expected Environment Variables** (`.env` file)

*   `'AP_DATA_RAW'`: Path to raw data
*   `'AP_DATA_PREPROCESSED'`: Path to preprocessed data


<br>

**Expected project structure**

```
└── Data
   ├── preprocessed
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── prompts_templates
   │   ├── template-task-agnostic-no-instructions-few-shot-en.jinja
   │   ├── template-task-specific-instructions-few-shot.jinja
   │   ├── template-task-specific-instructions-few-shot-json.jinja
   │   ├── template-task-specific-instructions-zero-shot.jinja
   │   ├── template-task-specific-meta-prompt-few-shot.jinja
   │   └── template-task-specific-roleplay-few-shot.jinja
   ├── raw
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── results
   │   ├── deepseek-v3.2
   │   ├── gemma2-9b-it
   │   ├── qwen3.5-9b
   │   ├── boto-9b-it
   │   └── multilingual-sentiment-analysis
   └── staging_area
```

<br>

**References**:
```
@misc{maas_2011_learning,
	title        = {Learning Word Vectors for Sentiment Analysis},
	author       = {Maas, Andrew L. and Daly, Raymond E. and Pham, Peter T. and others},
	year         = 2011,
	month        = {06},
	booktitle    = {ACLWeb},
	publisher    = {Association for Computational Linguistics},
	pages        = {142–150},
	urldate      = {2024-05-11},
	howpublished = {\url{https://aclanthology.org/P11-1015}}
}

@inproceedings{vargas_2020_identifying,
	title        = {Identifying Fine-Grained Opinion and Classifying Polarity on Coronavirus Pandemic},
	author       = {Vargas, Francielle Alves  and Sanches, Rodolfo and Rocha, Pedro Regattieri},
	year         = 2020,
	month        = 10,
	booktitle    = {Proceedings of the 9th Brazilian Conference on Intelligent Systems (BRACIS 2020)},
	publisher    = {Springer-Verlag},
	pages        = {511–520},
	doi          = {10.1007/978-3-030-61377-8_35},
	urldate      = {2024-05-11},
	note         = {\url{https://doi.org/10.1007/978-3-030-61377-8_35}}
}


@article{freitas_2014_sparkling,
	title        = {Sparkling vampire... lol! annotating opinions in a book review corpus},
	author       = {Freitas, Cláudia and Motta, Eduardo and Milidiú, Ruy  and others},
	year         = 2014,
	journal      = {New language technologies and linguistic research: a two-way Road},
	pages        = {128--146}
}

@inproceedings{brum_2018_building,
	title        = {{Building a Sentiment Corpus of Tweets in Brazilian Portuguese}},
	author       = {Henrico Brum and Maria das Gra\c{c}as Volpe Nunes},
	year         = 2018,
	month        = {May 7-12, 2018},
	booktitle    = {Proceedings of the Eleventh International Conference on Language Resources and Evaluation (LREC 2018)},
	publisher    = {European Language Resources Association (ELRA)},
	address      = {Miyazaki, Japan},
	isbn         = {979-10-95546-00-9},
	editor       = {Nicoletta Calzolari (Conference chair) and Khalid Choukri and Christopher Cieri and Thierry Declerck and Sara Goggi and Koiti Hasida and Hitoshi Isahara and Bente Maegaard and Joseph Mariani and HÚlŔne Mazo and Asuncion Moreno and Jan Odijk and Stelios Piperidis and Takenobu Tokunaga},
	language     = {english}
}
```


# Installations

Installing packages.

In [ ]:
! pip install transformers datasets

# 00 - Environment Setup

Load environment variables from `.env` file.








In [ ]:
from dotenv import load_dotenv
load_dotenv()

# 01 - Imports

Import the necessary libraries.

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import json
import requests


from sklearn.model_selection import train_test_split
from datasets import load_dataset
from pathlib import Path

# 02 - Constants
In this stage, two constants `RAW_DATA_ROOT_PATH` and `PREPROCESSED_DATA_ROOT_PATH` are defined.

The constant `RAW_DATA_ROOT_PATH` will store the path to the folder where the raw data is stored.

The constant `PREPROCESSED_DATA_ROOT_PATH` stores the path to the folder where the preprocessed data will be saved.


In [ ]:
RAW_DATA_ROOT_PATH = os.getenv("AP_DATA_RAW")
PREPROCESSED_DATA_ROOT_PATH = os.getenv("AP_DATA_PREPROCESSED")


# 03 - Functions

Definition of the functions that will be used during data preprocessing.

### 03.01 - General purpose functions

In [ ]:
def check_str_or_path(path:str|Path)->bool:
    """
    Check if the given path is a string or a Path object.

    Args:
        path (str | Path): The path to check.

    Returns:
        bool: True if the path is either a string or a Path object, False otherwise.
    """
    if isinstance(path, Path) or isinstance(path, str):
        return True
    else:
       return False

def check_file_exists(path:str|Path)->bool:
    """
    Check if a file exists at the given path.

    Args:
        path (str | Path): The path to the file.

    Returns:
        bool: True if the file exists, False otherwise.
    """
    if check_str_or_path(path):
        return os.path.isfile(path)

def check_directory_exists(path:str|Path)->bool:
    """
    Check if a directory exists at the given path.

    Args:
        path (str | Path): The path to the directory.

    Returns:
        bool: True if the directory exists, False otherwise.
    """
    if check_str_or_path(path):
        return os.path.isdir(path)

def create_directory(path:str|Path)->Path:
    """
    Create a directory at the given path if it does not already exist.

    Args:
        path (str | Path): The path where the directory will be created.

    Returns:
        Path: The Path object of the created directory if successful, or a message indicating that the directory already exists.
    """
    if check_str_or_path(path) and not check_directory_exists(path=path):
        os.mkdir(path)
        return Path(path)
    else:
        return f"Directory {path} already exists"

### 03.02 - Data Pipeline Functions

In [ ]:
def adjust_columns_names(df:pd.DataFrame, mapping:dict)->pd.DataFrame:
    """
    Adjust the column names of a DataFrame based on a provided mapping.

    Args:
        df (pd.DataFrame): The DataFrame whose columns need to be renamed.
        mapping (dict): A dictionary where keys are old column names and values are new column names.

    Returns:
        pd.DataFrame: The DataFrame with renamed columns.
    """
    return df.rename(columns=mapping)

def adjust_labels(df:pd.DataFrame,  label_column_name:str, mapping:dict)->pd.DataFrame:
    """
    Adjust the labels in a specified column of a DataFrame based on a provided mapping.

    Args:
        df (pd.DataFrame): The DataFrame containing the labels to be adjusted.
        label_column_name (str): The name of the column containing the labels to be adjusted.
        mapping (dict): A dictionary where keys are old labels and values are new labels.

    Returns:
        pd.DataFrame: The DataFrame with adjusted labels.
    """
    df[label_column_name] = df[label_column_name].map(mapping)
    return df

def drop_unused_labels(df:pd.DataFrame, label_column_name:str, condition:list, condition_type:str)->pd.DataFrame:
    """
    Drop unused labels from a specified column of a DataFrame based on a condition.

    Args:
        df (pd.DataFrame): The DataFrame containing the labels to be dropped.
        label_column_name (str): The name of the column containing the labels to be dropped.
        condition (list): A list of labels to keep or drop.
        condition_type (str): The type of condition, either "keep" or "drop".

    Returns:
        pd.DataFrame: The DataFrame with unused labels dropped.
    """
    if condition_type == "keep":
        df = df[df[label_column_name].isin(condition)].reset_index(drop=True)

    else:
        df = df[~df[label_column_name].isin(condition)].reset_index(drop=True)

    return df

def drop_unused_columns(df:pd.DataFrame, column_name:str|list, condition_type:str)->pd.DataFrame:
    """
    Drop unused columns from a DataFrame based on a condition.

    Args:
        df (pd.DataFrame): The DataFrame containing the columns to be dropped.
        column_name (str | list): The name or list of names of the columns to be dropped.
        condition_type (str): The type of condition, either "keep" or "drop".

    Returns:
        pd.DataFrame: The DataFrame with unused columns dropped.
    """
    if not isinstance(column_name, list):
        column_name = list(column_name)

    if condition_type == "keep":
        df = df[column_name]

    else:
         df = df.drop(columns=column_name, axis=1)

    return df


def random_sample(df:pd.DataFrame, sample_size:int, random_state:int=42, **kwargs)->pd.DataFrame:
    """
    Drop unused columns from a DataFrame based on a condition.

    Args:
        df (pd.DataFrame): The DataFrame containing the columns to be dropped.
        sample_size (int) : The size of the sample.
        random_state (int) : The random state for reproducibility. Default is 42.

    Returns:
        pd.DataFrame: The DataFrame with unused columns dropped.
    """
    df = df.sample(n=sample_size, random_state=random_state)
    return df

def create_train_test_split(df:pd.DataFrame, features=list|str, target=str)->dict:
    """
    Create train and test splits from a DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame to be split.
        features (list | str): The list of feature columns or a single feature column.
        target (str): The target column.
    Returns:
        dict: A dictionary containing the train and test splits.
    """
    if not isinstance(features, list):
        features_list = [features]

    if not isinstance(target, list):
        target_list = [target]

    features = df[features_list]
    target = df[target_list]

    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
    train = pd.concat([X_train, y_train], axis=1)
    test = pd.concat([X_test, y_test], axis=1)
    return {"train":train, "test":test}

def persist_splits_as_csv(split:dict, directory_path:str|Path, file_name:str)->None:
    """
    Persist train and test splits as CSV files in a specified directory.

    Args:
        split (dict): A dictionary containing the train and test splits.
        directory_path (str | Path): The path to the directory where the CSV files will be saved.
        file_name (str): The base name for the CSV files.

    Returns:
        None
    """
    if not isinstance(directory_path, Path):
        directory_path = Path(directory_path)

    if not check_directory_exists(directory_path):
        create_directory(directory_path)

    for key in split.keys():
        file_full_path = directory_path.joinpath(f"{file_name}_{key}.csv")
        split[key].to_csv(f"{file_full_path}", index=False, encoding="utf-8")
    return

## 04 - Datasets

Datasets preprocessing.

## 04.01 - Binary Classification

### 04.01.01 - IMDB_PT

**Reference**
<br>
[Maas, A. L., Daly, R. E., Pham, P. T., et al. (2011). Learning word vectors for
sentiment analysis.](https://aclanthology.org/P11-1015)

[Pires, R., Abonizio, H., Almeida, T. S., et al. (2023). Sabiá: Portuguese large
language models. In Naldi, M. C. and Bianchi, R. A. C., editors, Lecture Notes
in Computer Science, page 226–240. Springer Nature Switzerland.
](https://arxiv.org/abs/2304.07880)


<br>

**Dataset Link**
<br>
[IMDB_PT](https://huggingface.co/datasets/maritaca-ai/imdb_pt)

#### Defining Paths

In [ ]:
file_name="imdb-pt"
imdb_raw_path = RAW_DATA_ROOT_PATH + f"/{file_name}"
imdb_preprocessed_path = PREPROCESSED_DATA_ROOT_PATH + f"/{file_name}"

if not check_directory_exists(path=imdb_raw_path):
    create_directory(path=imdb_raw_path)

if not check_directory_exists(path=imdb_preprocessed_path):
    create_directory(path=imdb_preprocessed_path)

#### Reading data

In [ ]:
dataset = load_dataset("maritaca-ai/imdb_pt")
train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()

#### Pre-processing pipeline

In [ ]:
label_mapping = {
    0: -1,
    1:1
}

train = train.pipe(adjust_labels, label_column_name='label', mapping=label_mapping)
test = (test.pipe(adjust_labels, label_column_name='label', mapping=label_mapping)
            .pipe(random_sample, sample_size=1_500))

splits = {
    "train": train,
    "test":test
}

persist_splits_as_csv(
    split=splits,
    directory_path=imdb_preprocessed_path,
    file_name=file_name)

#### Datasets metadata

In [ ]:
print(f'Train dataset size: {splits["train"].shape}\n\n')

print(f'Test dataset size: {splits["test"].shape}\n\n')

print(f'Label distribution Train dataset:\n {splits["train"]["label"].value_counts(dropna=False, normalize=True)}')

### 04.01.02 - OPCovidBR

**Reference**
<br>
[Vargas, F. A., Sanches, R., and Rocha, P. R. (2020). Identifying fine-grained opinion and classifying polarity on coronavirus pandemic. In Proceedings of the 9th Brazil ian Conference on Intelligent Systems (BRACIS 2020), page 511–520. Springer-Verlag. https://doi.org/10.1007/978-3-030-61377-8_35](https://doi.org/10.1007/978-3-030-61377-8_35)

<br>

**Dataset Link**
<br>
[OPCovidBR](https://github.com/franciellevargas/OPCovidBR/blob/master/data/opcovid-br/opcovidbr.csv)

#### Defining Paths

In [ ]:
file_name="opcovidbr"
opcovidbr_raw_path = RAW_DATA_ROOT_PATH +f"/{file_name}"
opcovidbr_preprocessed_path = PREPROCESSED_DATA_ROOT_PATH + f"/{file_name}"


if not check_directory_exists(path=opcovidbr_raw_path):
    create_directory(path=opcovidbr_raw_path)

if not check_directory_exists(path=opcovidbr_preprocessed_path):
    create_directory(path=opcovidbr_preprocessed_path)

#### Reading data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/franciellevargas/OPCovidBR/master/data/opcovid-br/opcovidbr.csv")
df.info()

#### Pre-processing pipeline

In [ ]:
column_name_mapping = {
    'twitter':'text',
    'polarity':'label'
}

splits = (
    df.pipe(drop_unused_columns, column_name=["twitter", "polarity"], condition_type="keep")
    .pipe(drop_unused_labels, label_column_name='polarity', condition=[-1, 1], condition_type="keep")
    .pipe(adjust_columns_names, mapping=column_name_mapping)
    .pipe(create_train_test_split, features="text", target="label")
    )


persist_splits_as_csv(
    split=splits,
    directory_path=opcovidbr_preprocessed_path,
    file_name=file_name)

#### Datasets metadata

In [ ]:
print(f'Train dataset size: {splits["train"].shape}\n\n')

print(f'Test dataset size: {splits["test"].shape}\n\n')

print(f'Label distribution Train dataset:\n {splits["train"]["label"].value_counts(dropna=False, normalize=True)}')

## 04.02 - Multiclass Classification

### 04.02.01 - ReLI

**Reference**
<br>
Freitas, C., Motta, E., Milidiú, R., et al. (2014). Sparkling vampire... lol! annotat ing opinions in a book review corpus. New language technologies and linguistic research: a two-way Road, pages 128–146

<br>

**Dataset Link**
<br>
[ReLI](https://www.linguateca.pt/Repositorio/ReLi/)

#### Defining Paths

In [ ]:
file_name="reli"
reli_raw_path = RAW_DATA_ROOT_PATH + f"/{file_name}"
reli_preprocessed_path = PREPROCESSED_DATA_ROOT_PATH + f"/{file_name}"

if not check_directory_exists(path=reli_raw_path):
    create_directory(path=reli_raw_path)

if not check_directory_exists(path=reli_preprocessed_path):
    create_directory(path=reli_preprocessed_path)

#### Reading data

In [ ]:
!wget  https://raw.githubusercontent.com/pedrobalage/ReLi_Experiments/master/ReLi.py

In [ ]:
from ReLi import ReLiCorpusReader

reli_raw_path = RAW_DATA_ROOT_PATH + '/ReLI'
reli_preprocessed_path = PREPROCESSED_DATA_ROOT_PATH + "/reli"
file_name="reli"

reli_txt_files_paths =  glob.glob(f"{reli_raw_path}/**.txt")


corpus = ReLiCorpusReader(path=reli_raw_path)

lista = []
for book in corpus.keys():
    for review in corpus[book]:
        score = corpus[book][review].get('score', None)
        if corpus[book][review].get('sentences'):
            for sentence in corpus[book][review]['sentences']:
                phrase = " ".join(corpus.words_sentence(sentence))
                sentiment =  1 if sentence[0][4] == '+' else -1 if sentence[0][4] == "-" else 0
                lista.append([book, review, score, phrase, sentiment])


df = pd.DataFrame(lista, columns=['book', 'review', 'score','phrase', 'sentiment'])
df.info()

#### Pre-processing pipeline

In [ ]:
column_name_mapping ={
    "phrase":"text",
    "sentiment": "label"
}


splits = (
    df.pipe(drop_unused_columns, column_name=['phrase', 'sentiment'], condition_type="keep")
    .pipe(adjust_columns_names, mapping=column_name_mapping)
    .pipe(random_sample, sample_size=7_500)
    .pipe(create_train_test_split, features="text", target="label")
    )


persist_splits_as_csv(
    split=splits,
    directory_path=reli_preprocessed_path,
    file_name=file_name)

#### Datasets metadata

In [ ]:
print(f'Train dataset size: {splits["train"].shape}\n\n')

print(f'Test dataset size: {splits["test"].shape}\n\n')

print(f'Label distribution Train dataset:\n {splits["train"]["label"].value_counts(dropna=False, normalize=True)}')

### 04.02.02 - TweetSentBr

**Reference**
<br>
[Brum, H. and das Graças Volpe Nunes, M. (2018). Building a Sentiment Corpus
of Tweets in Brazilian Portuguese. In chair), N. C. C., Choukri, K., Cieri, C.,
Declerck, T., Goggi, S., Hasida, K., Isahara, H., Maegaard, B., Mariani, J., Mazo,H., Moreno, A., Odijk, J., Piperidis, S., and Tokunaga, T., editors, Proceedings of the Eleventh International Conference on Language Resources and Evaluation (LREC 2018), Miyazaki, Japan. European Language Resources Association
(ELRA)](https://aclanthology.org/L18-1658/)
<br>


**Dataset Link**
<br>
[TweetSentBr](https://huggingface.co/datasets/eduagarcia/tweetsentbr_fewshot)

#### Defining Paths

In [ ]:
file_name="tweetsentbr"
tweetsentbr_raw_path = RAW_DATA_ROOT_PATH + f"/{file_name}"
tweetsentbr_preprocessed_path = PREPROCESSED_DATA_ROOT_PATH + f"/{file_name}"


if not check_directory_exists(path=tweetsentbr_raw_path):
    create_directory(path=tweetsentbr_raw_path)

if not check_directory_exists(path=tweetsentbr_preprocessed_path):
    create_directory(path=tweetsentbr_preprocessed_path)

#### Reading data

In [ ]:
dataset = load_dataset("eduagarcia/tweetsentbr_fewshot")
train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()

In [ ]:
train["label"].value_counts()

In [ ]:
test["label"].value_counts()

In [ ]:
indexes = test.groupby('label').apply(lambda x: x.sample(500, random_state=42)).index.get_level_values(1).to_list()
test = test.loc[indexes]
test = test.sort_index()

#### Pre-processing pipeline

In [ ]:
label_mapping = {
    'Positive': 1,
    'Neutral': 0,
    'Negative': -1,
}

column_name_mapping = {
    'sentence':'text'
    }

train = (
    train.pipe(drop_unused_columns, column_name=['id'], condition_type="drop")
    .pipe(adjust_labels, label_column_name='label', mapping=label_mapping)
    .pipe(adjust_columns_names, mapping=column_name_mapping)
    )



test = (
    test.pipe(drop_unused_columns, column_name=['id'], condition_type="drop")
    .pipe(adjust_labels, label_column_name='label', mapping=label_mapping)
    .pipe(adjust_columns_names, mapping=column_name_mapping)
    )

splits = {
    "train": train,
    "test":test
}

persist_splits_as_csv(
    split=splits,
    directory_path=tweetsentbr_preprocessed_path,
    file_name=file_name)

#### Datasets metadata

In [ ]:
print(f'Train dataset size: {splits["train"].shape}\n\n')

print(f'Test dataset size: {splits["test"].shape}\n\n')

print(f'Label distribution Train dataset:\n {splits["train"]["label"].value_counts(dropna=False, normalize=True)}')

print(f'Label distribution Test dataset:\n {splits["test"]["label"].value_counts(dropna=False, normalize=True)}')